# Kiểm chứng và kết luận MixStyle — 10/11/2026

Notebook này hoàn tất phân tích của 12 checkpoint và 57 capture ngoại bộ đã lưu.
Notebook 10 được giữ làm lịch sử: lần tổng hợp đầu lỗi do baseline có thêm dòng
ngưỡng mặc định 0,5. Phân tích hiện tại lọc đúng `source_calibrated`.
Mã runner cũ được khôi phục và đối chiếu đúng SHA256 trong `provenance/`.

Không chọn lại mô hình theo test. Probe và runtime dùng checkpoint đã đóng băng.
Đây là nghiên cứu hậu nghiệm, không thay kết quả chính của khóa luận.


In [1]:
from pathlib import Path
import sys, subprocess, json
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p/'code/scripts/run_mixstyle_study.py').is_file())
OUT = ROOT/'code/results/mixstyle_posthoc_2026-11-09'
def run(*args):
    result = subprocess.run([sys.executable, '-X', 'utf8', *args], cwd=ROOT,
                            text=True, encoding='utf-8', stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()
run('-m', 'pytest', 'code/tests/test_mixstyle.py', '-q',
    '--junitxml='+str(OUT/'mixstyle_tests_final.xml'))


...........                                                              [100%]
============================== warnings summary ===============================
code/tests/test_mixstyle.py::test_analysis_excludes_historical_default_threshold_rows
code/tests/test_mixstyle.py::test_analysis_excludes_historical_default_threshold_rows
code/tests/test_mixstyle.py::test_analysis_excludes_historical_default_threshold_rows
code/tests/test_mixstyle.py::test_analysis_excludes_historical_default_threshold_rows
  F:\00_Research\code\scripts\run_mixstyle_study.py:283: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
    d = pd.read_csv(unit/'metrics.csv', float_precision='round_trip'); check_metrics(d)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
11 passed, 4 warnings in 30.23s



In [2]:
# Recompute primary metrics and paired intervals from frozen predictions.
run('code/scripts/run_mixstyle_study.py', '--stage', 'summarize', '--output', str(OUT))


F:\00_Research\code\scripts\run_mixstyle_study.py:283: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  d = pd.read_csv(unit/'metrics.csv', float_precision='round_trip'); check_metrics(d)
F:\00_Research\code\scripts\run_mixstyle_study.py:283: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  d = pd.read_csv(unit/'metrics.csv', float_precision='round_trip'); check_metrics(d)
F:\00_Research\code\scripts\run_mixstyle_study.py:283: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  d = pd.read_csv(unit/'metrics.csv', float_precision='round_trip'); check_metrics(d)
F:\00_Research\code\scripts\run_mixstyle_study.py:283: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  d = pd.read_csv(unit/'metrics.csv', float_precision='round_trip'); check_metrics(d)
                        

In [3]:
# Reuse hash-verified completed secondary work; otherwise execute it.
run('code/scripts/complete_mixstyle_secondary.py')
run('code/scripts/report_mixstyle_results.py')


VERIFIED F:\00_Research\code\results\mixstyle_posthoc_2026-11-09\secondary\attempt_002



F:\00_Research\code\scripts\report_mixstyle_results.py:37: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  new = pd.read_csv(dest/'external_metrics.csv', float_precision='round_trip')
F:\00_Research\code\scripts\report_mixstyle_results.py:38: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  old = pd.read_csv(OLD/'external_metrics.csv', float_precision='round_trip')
KẾT QUẢ THỬ NGHIỆM BỔ SUNG MIXSTYLE — HOÀN TẤT 10/11/2026
Thí nghiệm hậu nghiệm; giữ nguyên kết quả và bản khóa luận đã bàn giao.
12 fit mới (4 set x 3 seed), 24 baseline tái sử dụng; W64/content/file-order.
Có thêm một pilot phần mềm 2 epoch trên 80 cửa sổ; không tính vào 12 fit nghiên cứu.
Ngưỡng chỉ chọn từ validation nguồn với FAR budget 100/100000.
Mọi điểm dưới đây trung bình seed trong set, rồi trọng số ngang nhau giữa set.

Điều kiện A
  raw_transformer: Recall 61.67%; FAR 0.12%; AP 0.7312; F1 0.7258
  str

In [4]:
import pandas as pd
display(pd.read_csv(OUT/'analysis/six_method_equal_set.csv').query("condition == 'B'"))
display(pd.read_csv(OUT/'analysis/probe_means.csv').query("control == 'learned'"))
display(pd.read_csv(OUT/'analysis/runtime_remeasured.csv'))
print(json.loads((OUT/'analysis/final_report.json').read_text(encoding='utf-8')))
print((OUT/'analysis/BAO_CAO_MIXSTYLE.txt').read_text(encoding='utf-8'))


,method,condition,Recall,FAR_per_100k,Attack_F1,AP
1,assoc_rules_crysys,B,0.000000,0.000000,0.000000,0.059093
5,canbertplus,B,0.639479,64843.848157,0.059262,0.097000
9,canlp,B,0.285434,29695.884205,0.032880,0.054229
13,raw_transformer,B,0.854149,65301.408666,0.300019,0.378089
17,structural_transformer,B,0.738296,52461.455675,0.243478,0.555448
21,structural_mixstyle,B,0.819440,64538.712463,0.229848,0.581507


,representation,target,control,balanced_accuracy
1,raw,attack,learned,0.641900
5,raw,vehicle,learned,0.992529
9,structural,attack,learned,0.766918
13,structural,vehicle,learned,0.948833
17,structural_mixstyle,attack,learned,0.773671
21,structural_mixstyle,vehicle,learned,0.952712


,device,representation,n,p50_ms,p95_ms,mean_ms,checkpoint_manifest
0,cpu,raw,900,4.48085,6.007070,4.620537,F:\00_Research\code\results\comparison_gpu4_re...
1,cpu,structural,900,4.00075,5.032610,4.096231,F:\00_Research\code\results\comparison_gpu4_re...
2,cpu,structural_mixstyle,900,3.78505,4.426655,3.835716,F:\00_Research\code\results\mixstyle_posthoc_2...
3,cuda,raw,900,3.96165,5.786070,4.211416,F:\00_Research\code\results\comparison_gpu4_re...
4,cuda,structural,900,3.83620,4.616685,3.949724,F:\00_Research\code\results\comparison_gpu4_re...
5,cuda,structural_mixstyle,900,3.82315,4.596790,3.933348,F:\00_Research\code\results\mixstyle_posthoc_2...


{'status': 'VERIFIED_COMPLETE', 'new_fits': 12, 'reused_baselines': 24, 'external_captures': 57, 'external_metric_rows': 513, 'primary_baseline_aggregates_unchanged': True, 'post_hoc': True, 'inference_labels_used': False, 'primary_comparisons': [{'condition': 'B', 'baseline': 'raw_transformer', 'recall_delta_pp': -3.4708580831573466, 'far_delta_pp': -0.7626962032670271, 'interpretation': 'RECALL_DECREASE', 'sets_with_higher_recall': 1}, {'condition': 'B', 'baseline': 'structural_transformer', 'recall_delta_pp': 8.11449309044342, 'far_delta_pp': 12.0772567876303, 'interpretation': 'INCREASED_FALSE_ALARMS_TRADEOFF', 'sets_with_higher_recall': 3}], 'unperformed_secondary_studies': [], 'completion_date': '2026-11-10', 'secondary_manifest_sha256': '48e978e3be83224b38b5b200abc69640edac490b26b0b1e56ab5da7bb229f779', 'aggregation_fix_provenance': 'provenance/aggregation_fix.json', 'analysis_runner_sha256': '471530e638d92b665a376db037779d1e98d3d4bb26fa4e4e107b4e3d83722ef5', 'report_script_sha2